In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import joblib
import random
from tqdm import tqdm
from collections import defaultdict

from utils.beta_oracle import load_beta_oracle, oracle_eval, noise_variance

# 對各個配方系列按照設定隨機生成數值
def generate_key_data(config, target_sum=100):
    multiplier = 10  # 處理小數點後一位
    target_int = int(target_sum * multiplier)

    # print('生成 adj_config')
    # 初始化：確保 min >= 0.1 (即放大後的 1)
    max_sum = 0
    while max_sum < target_int:
        adj_config = {}
        for k, v in config.items():
            # {'min', 'max', 'num_non_zero'}
            cfg = {}

            # 根據不為0欄位的設定值抽選不為0欄位的數量
            cfg['num_non_zero'] = random.randint(v['non_zero_min'], v['non_zero_max'])

            if cfg['num_non_zero'] > 0:
                cfg['min'] = max(1, int(v['min'] * multiplier))
                cfg['max'] = int(v['max'] * multiplier)
            else:
                cfg['min'] = 0
                cfg['max'] = 0
            
            adj_config[k] = cfg
        max_sum = sum([v['max'] for k,v in adj_config.items()])
        

    res_int = {k: v['min'] for k, v in adj_config.items()}
    remaining = target_int - sum(res_int.values())

    if remaining < 0:
        raise ValueError('設定的最小值總和已超過目標 {}'.format(target_sum))
    
    # 選出允許有子欄位的 key
    keys = [k for k in adj_config.keys() if adj_config[k]['num_non_zero'] > 0]
    
    while remaining > 0:
        # 計算所有 K 的剩餘空間可以分配
        space_dic = { k: adj_config[k]['max'] - res_int[k] for k in keys}
        have_space_keys = [k for k,v in space_dic.items() if v > 0]

        if len(have_space_keys) > 0:
            # 隨機選擇配方大類
            k = random.choice(have_space_keys)
            space = space_dic[k]
            add_val = random.randint(1, min(remaining, space))
            res_int[k] += add_val
            remaining -= add_val
            
            # space = adj_config[k]['max'] - res_int[k]
            # if space > 0:
            #     add_val = random.randint(1, min(remaining, space))
            #     res_int[k] += add_val
            #     remaining -= add_val

    value_set = {k: v / multiplier for k, v in res_int.items()}
    nun_zero_set = {k: v['num_non_zero'] for k, v in adj_config.items()}
    output = {
        'value_set':value_set,
        'nun_zero_set': nun_zero_set
    }
    # 回傳以 Key 為單位的結果 (已轉回浮點數)
    is_finish = True
    return output
# 將 Key 值分配給子欄位 (總和等於 Key 值, 允許為 0) ---
def generate_sub_col_data(key_data, cols):
    key_value_set = key_data['value_set']
    num_zero_set = key_data['nun_zero_set']
    # 分類子欄位
    col_mapping = defaultdict(list)
    for col in cols:
        prefix = col[:2]
        col_mapping[prefix].append(col)

    final_values = {}

    for prefix, total_val in key_value_set.items():
        sub_cols = col_mapping.get(prefix, [])
        if not sub_cols:
            continue
        
        # # 轉成整數處理以確保精確度 (10倍)
        # total_int = int(round(total_val * 10))
        if len(sub_cols) == 1:
            final_values[sub_cols[0]] = total_val
        else:
            non_zero = num_zero_set[prefix]
                
            # 如果 non_zero 數量比 len(sub_cols) 多就將 non_zero 設定成 len(sub_cols)
            non_zero = non_zero if non_zero < len(sub_cols) else len(sub_cols)
            non_zero_col = random.sample(sub_cols, non_zero) # 抽選不為 0 的 col

            # 隨機產生數字
            values = (np.random.dirichlet(alpha=[0.5]*non_zero, size=1)[0] * total_val).tolist()
            cols_values = { non_zero_col[i]:values[i] for i in range(len(non_zero_col))}

            for sub_col in sub_cols:
                final_values[sub_col] = cols_values.get(sub_col, 0.)

        # print(prefix, total_val)
                
    return final_values

# 基準生成器
def design_dirichlet(n:int, beta_csv_path:str) -> dict:
    '''
    使用 Dirichlet 分佈進行採樣（α=1 為均勻分佈）

    :param n: 產生的資料集數量
    :type n: int
    :param beta_csv_path: oracle function 的係數 csv
    :type beta_csv_path: str

    :return 
    :dataset: dict
    :describe => 
    :dataset['X']: tensor: 資料集的 X
    :dataset['Y']: tensor: 資料集的 Y
    :dataset['feature_cols']: list[str]: 資料集 X 的 column name
    :dataset['target_cols']: list[str]: 資料集 Y 的 column name
    '''
    oracle = load_beta_oracle(beta_csv_path, device=device, dtype=dtype)

    q = len(oracle["feature_cols"])

    # 基準生成器: 隨機創建 100 筆符合 Dirichlet 分佈的資料
    X = torch.distributions.Dirichlet(torch.ones(q, device=device, dtype=dtype)).sample((n,))
    Y = oracle_eval(X, oracle)

    dataset = {
        'X': X,
        'Y': Y,
        'feature_cols': oracle["feature_cols"], 
        'target_cols': oracle['target_cols']
    }

    return dataset

# 稀疏感知生成器 (Sparsity-aware):
def design_sparse(n:int, beta_csv_path:str, config:dict, max_sum:int, x_standerize:bool=True):
    '''
    :param n: 產生的資料集數量
    :type n: int
    :param beta_csv_path: oracle function 的係數 csv
    :type beta_csv_path: str
    :param config: 配方各類別的生成資料 (每個類別的最大值、最小值、非零欄位數量的最小值及最大值)
    :type config: dict
    :example config:
    config = {
        'AA': { 'min': 0,  'max': 1,  'non_zero_min': 0, 'non_zero_max': 3 },
        'AW': { 'min': 0,  'max': 1,  'non_zero_min': 0, 'non_zero_max': 3 },
        'AX': { 'min': 0,  'max': 10, 'non_zero_min': 0, 'non_zero_max': 3 },
        'CM': { 'min': 0,  'max': 5,  'non_zero_min': 0, 'non_zero_max': 3 },
        'CP': { 'min': 0,  'max': 1,  'non_zero_min': 0, 'non_zero_max': 3 },
        'FR': { 'min': 0,  'max': 20, 'non_zero_min': 0, 'non_zero_max': 3 },
        'GF': { 'min': 0,  'max': 51, 'non_zero_min': 0, 'non_zero_max': 3 },
        'MF': { 'min': 0,  'max': 20, 'non_zero_min': 0, 'non_zero_max': 3 },
        'PR': { 'min': 30, 'max': 90, 'non_zero_min': 1, 'non_zero_max': 3 },
        'SS': { 'min': 0,  'max': 10, 'non_zero_min': 0, 'non_zero_max': 3 },
    }
    :param max_sum: 單筆資料 X 值得總和
    :type max_sum: int

    :param x_standerize: X 數值是否要標準化成 0~1，需要考慮 oracle function 的定義域是否為 0~1，如果是的話這個參數請設定成 True
    :type x_standerize: bool

    :return 
    :dataset: dict
    :describe => 
    :dataset['X']: tensor: 資料集的 X
    :dataset['Y']: tensor: 資料集的 Y
    :dataset['feature_cols']: list[str]: 資料集 X 的 column name
    :dataset['target_cols']: list[str]: 資料集 Y 的 column name
    '''
    # 讀取 oracle function
    oracle = load_beta_oracle(beta_csv_path, device=device, dtype=dtype)

    # 配方欄位
    formula_cols = oracle["feature_cols"]

    # 配方類別
    formula_class_ls = list(set([c[:2] for c in formula_cols]))
    formula_class_ls.sort()

    # 配方類別設定
    formula_cfg = { c:config[c] for c in formula_class_ls}

    formula_samples = []
    t =tqdm(range(1000))
    for i in t:
        key_res = generate_key_data(formula_cfg, target_sum=max_sum)
        col_res = generate_sub_col_data(key_res, formula_cols)

        # 確保數值總和=max_sum
        ratio = max_sum/sum([v for k,v in col_res.items()])
        for k in col_res.keys():
            col_res[k] = col_res[k] * ratio

        formula_samples.append(col_res)

    formula_samples = pd.DataFrame(formula_samples)
    X = np.array(formula_samples)

    # 要套用庭軒的 oracle model，X 的總和要=1，所以所有的 X 要除以 max_sum (100)
    if x_standerize:
        X = X / max_sum
        
    X = torch.tensor(X, device=device)

    # 透過 oracle model輸入 X 後生成 Y
    Y = oracle_eval(X, oracle)

    dataset = {
        'X': X,
        'Y': Y,
        'feature_cols': oracle["feature_cols"],
        'target_cols': oracle['target_cols']
    }
    return dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64

In [ ]:
# 資料集生成位置
base_dataset_save_path = '/workspaces/BO_EXPERIMENTS/src/datasets/base_dataset.pt'
sparse_dataset_save_path = '/workspaces/BO_EXPERIMENTS/src/datasets/sparse_dataset.pt'

# 讀取 oracle model
beta_path = '/workspaces/BO_EXPERIMENTS/src/datasets/beta1.csv'
# oracle = load_beta_oracle(beta_path, device=device, dtype=dtype)

In [ ]:
# 生成基準資料
base_dataset = design_dirichlet(n=100, beta_csv_path=beta_path)
torch.save(base_dataset, base_dataset_save_path)

# 用配方限制的方法隨機生成配方
### PBTc 配方限制
![!image](https://gitlab-friends.ccp.com.tw/issue_dashboard/chemopt/uploads/badbe3e664a10d0e4da352ce325c00ec/image.png)

In [ ]:
# 稀疏感知生成器 (Sparsity-aware)：生成 1000 筆X資料

# 各類別的最小值(min)、最大值(max) 以及非0欄位的數量(non_zero)
rd_config = {
    'AA': { 'min': 0,  'max': 1,  'non_zero_min': 0, 'non_zero_max': 3 },
    'AW': { 'min': 0,  'max': 1,  'non_zero_min': 0, 'non_zero_max': 3 },
    'AX': { 'min': 0,  'max': 10, 'non_zero_min': 0, 'non_zero_max': 3 },
    'CM': { 'min': 0,  'max': 5,  'non_zero_min': 0, 'non_zero_max': 3 },
    'CP': { 'min': 0,  'max': 1,  'non_zero_min': 0, 'non_zero_max': 3 },
    'FR': { 'min': 0,  'max': 20, 'non_zero_min': 0, 'non_zero_max': 3 },
    'GF': { 'min': 0,  'max': 51, 'non_zero_min': 0, 'non_zero_max': 3 },
    'MF': { 'min': 0,  'max': 20, 'non_zero_min': 0, 'non_zero_max': 3 },
    'PR': { 'min': 30, 'max': 90, 'non_zero_min': 1, 'non_zero_max': 3 },
    'SS': { 'min': 0,  'max': 10, 'non_zero_min': 0, 'non_zero_max': 3 },
}

sparse_dataset = design_sparse(100, beta_csv_path=beta_path, config=rd_config, max_sum=100, x_standerize=True)
torch.save(sparse_dataset, sparse_dataset_save_path)


In [ ]:
# n:int = 100
# beta_csv_path:str = beta_path
# config:dict = rd_config
# max_sum:int = 100
# x_standerize:bool=True

# # 讀取 oracle function
# oracle = load_beta_oracle(beta_csv_path, device=device, dtype=dtype)

# # 配方欄位
# formula_cols = oracle["feature_cols"]

# # 配方類別
# formula_class_ls = list(set([c[:2] for c in formula_cols]))
# formula_class_ls.sort()

# # 配方類別設定
# formula_cfg = { c:config[c] for c in formula_class_ls}

# formula_samples = []
# t =tqdm(range(1000))
# for i in t:
#     key_res = generate_key_data(formula_cfg, target_sum=max_sum)
#     col_res = generate_sub_col_data(key_res, formula_cols)

#     # 確保數值總和=max_sum
#     ratio = max_sum/sum([v for k,v in col_res.items()])
#     for k in col_res.keys():
#         col_res[k] = col_res[k] * ratio

#     formula_samples.append(col_res)

# formula_samples = pd.DataFrame(formula_samples)
# X = np.array(formula_samples)

# # 要套用庭軒的 oracle model，X 的總和要=1，所以所有的 X 要除以 max_sum (100)
# if x_standerize:
#     X = X / max_sum
    
# X = torch.tensor(X, device=device)

# # 透過 oracle model輸入 X 後生成 Y
# Y = oracle_eval(X, oracle)

# dataset = {
#     'X': X,
#     'Y': Y,
#     'feature_cols': oracle["feature_cols"],
#     'target_cols': oracle['target_cols']
# }

In [ ]:
# torch.sum(sparse_dataset['X'], axis = 1) == 1